# Contract Download Endpoint Audit

This notebook audits DLD/Dubai Now contract download endpoints for the five configured Emirates IDs.

It fetches contract history, filters contracts with status `Active`, `Pending`, or `Termination Request`, then checks:

- `downloadTenancyContract` for every filtered contract.
- `download` for `Active` and `Termination Request` contracts.

The Emirates ID ending in `0512` is processed first, followed by the other four configured users.

If any endpoint fails, the notebook writes a reproducible curl file and a response JSON file under `runs/contract_download_audit_<timestamp>/failures`.


## Configuration

Secrets are read from Colab userdata when available, otherwise from local `.env`.

Required secrets/env vars:

- `BASIC_AUTH`
- `CONSUMER_ID`
- `IDS_BASE_URL`
- `DLD_BASE_URL`
- `DLD_PROXY_URL`

Optional:

- `REQUEST_TIMEOUT_SECONDS`, default capped to `45` for this audit.


In [ ]:
# @title Configuration and environment
import base64
import csv
import json
import os
import re
import time
from datetime import datetime
from pathlib import Path

import requests

try:
    from google.colab import userdata
except Exception:
    userdata = None

try:
    from notebook_operator_utils import choose_option
except Exception:
    choose_option = None

# Required order: 0512 first, then the other 4 known users.
OWNER_EMIRATES_IDS = [
    "784195279540512",
    "784199515347708",
    "784199668638416",
    "784197265140323",
    "784198721116758",
]

REQUEST_TIMEOUT_SECONDS = 45
STREAM_SAMPLE_BYTES = 512 * 1024
CURRENT_BEARER_TOKEN = None


def load_local_env(path=".env"):
    env_path = Path(path)
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_local_env()


def get_secret(name, default=None):
    if userdata is not None:
        value = userdata.get(name)
        if value is not None:
            return value
    return os.environ.get(name, default)


def require_secret(name):
    value = get_secret(name)
    if value is None or str(value).strip() == "":
        raise RuntimeError(f"Missing required secret/env var: {name}")
    return str(value).strip()


BASIC_AUTH = require_secret("BASIC_AUTH")
CONSUMER_ID = require_secret("CONSUMER_ID")
IDS_BASE_URL = require_secret("IDS_BASE_URL").rstrip("/")
DLD_BASE_URL = require_secret("DLD_BASE_URL").rstrip("/")
DLD_PROXY_URL = require_secret("DLD_PROXY_URL").rstrip("/")

try:
    REQUEST_TIMEOUT_SECONDS = int(get_secret("REQUEST_TIMEOUT_SECONDS", REQUEST_TIMEOUT_SECONDS))
except Exception:
    pass

# Keep this audit bounded. If an endpoint stalls, we want a failure artifact instead of a hung notebook.
REQUEST_TIMEOUT_SECONDS = min(max(REQUEST_TIMEOUT_SECONDS, 15), 45)

RUN_DIR = Path("runs") / f"contract_download_audit_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
FAILURE_DIR = RUN_DIR / "failures"
RUN_DIR.mkdir(parents=True, exist_ok=True)
FAILURE_DIR.mkdir(parents=True, exist_ok=True)

print("Configured Emirates IDs, in processing order:")
for emirates_id in OWNER_EMIRATES_IDS:
    print("-", emirates_id)
print("Run folder:", RUN_DIR)
print("Request timeout seconds:", REQUEST_TIMEOUT_SECONDS)


## Operator Confirmation

Choose `Process all` for the normal full audit. Choose `Ask before each Emirates ID` when you want to skip one or more users without editing the configured ID list.

In [ ]:
# @title Operator confirmation

def text_choice(prompt, options, default):
    lookup = {str(i): value for i, (value, _) in enumerate(options, start=1)}
    lookup.update({value.lower(): value for value, _ in options})
    print(prompt)
    for i, (value, label) in enumerate(options, start=1):
        marker = " [default]" if value == default else ""
        print(f"{i}. {label}{marker}")
    while True:
        answer = input("Select option: ").strip().lower()
        if not answer:
            return default
        selected = lookup.get(answer)
        if selected:
            return selected
        print("Please choose one of the listed options.")


def ask_choice(prompt, options, default, title="Notebook option"):
    if choose_option is not None:
        return choose_option(
            prompt,
            [{"value": value, "label": label} for value, label in options],
            default=default,
            title=title,
        )
    return text_choice(prompt, options, default)


master_selection = ask_choice(
    "Process all configured Emirates IDs?",
    [("all", "Process all"), ("confirm_each", "Ask before each Emirates ID")],
    default="all",
    title="Contract download audit scope",
)
print("Selection:", master_selection)

## Shared Helpers

These helpers intentionally keep failure curl files unredacted so the API team can reproduce the failed request. Treat generated `.sh` files as sensitive.


In [ ]:
# @title Shared helpers

def safe_filename(value, max_len=100):
    return re.sub(r"[^a-zA-Z0-9_.=-]+", "_", str(value or "unknown"))[:max_len]


def shell_single_quote(value):
    return "'" + str(value).replace("'", "'\"'\"'") + "'"


def build_curl(method, url, headers=None, payload=None):
    line_continuation = " " + chr(92) + "\n"
    curl = f"curl --location --request {method.upper()} {shell_single_quote(url)}" + line_continuation
    for key, value in (headers or {}).items():
        curl += f"--header {shell_single_quote(f'{key}: {value}')}" + line_continuation
    if payload is not None:
        curl += f"--data-raw {shell_single_quote(json.dumps(payload, ensure_ascii=False))}"
    else:
        curl = curl[:-len(line_continuation)] if curl.endswith(line_continuation) else curl
    return curl


def decode_json_like(value):
    if isinstance(value, str):
        text = value.strip()
        if text.startswith("{") or text.startswith("["):
            try:
                return decode_json_like(json.loads(text))
            except Exception:
                return value
        return value
    if isinstance(value, dict):
        return {key: decode_json_like(inner_value) for key, inner_value in value.items()}
    if isinstance(value, list):
        return [decode_json_like(item) for item in value]
    return value


def response_payload(response):
    try:
        return decode_json_like(response.json())
    except Exception:
        return decode_json_like(response.text)


def compact_payload(value, max_chars=1200):
    if isinstance(value, str):
        text = value
    else:
        text = json.dumps(value, ensure_ascii=False, default=str)
    return text if len(text) <= max_chars else text[:max_chars] + "..."


def api_errors(payload):
    if not isinstance(payload, dict):
        return []
    errors = []
    response_code = payload.get("ResponseCode") or payload.get("responseCode")
    if response_code is not None and str(response_code).strip().lower() not in {"success", "ok", "200"}:
        errors.append(f"ResponseCode={response_code}")
    for key in ("ErrorMessage", "errorMessage", "Message", "message", "ReasonCode"):
        value = payload.get(key)
        if value and str(value).strip().lower() not in {"0", "success"}:
            errors.append(f"{key}={value}")
    for error in payload.get("ValidationErrorsList") or payload.get("validationErrorsList") or []:
        if not isinstance(error, dict):
            errors.append(str(error))
            continue
        error_number = error.get("ErrorNumber") or error.get("errorNumber")
        message_set = error.get("ErrorMessageSet") or error.get("errorMessageSet") or {}
        message = error.get("ErrorMessage") or error.get("errorMessage")
        if isinstance(message_set, dict):
            message = message or message_set.get("EnglishName") or message_set.get("englishName") or message_set.get("ArabicName")
        normalized_message = str(message or "").strip().lower()
        if error_number not in (None, 0, "0"):
            errors.append(f"ErrorNumber={error_number}; {message or compact_payload(error, 300)}")
        elif message and normalized_message not in {"no errors found", "no errors found."}:
            errors.append(str(message))
    return errors


def display_name(value):
    if isinstance(value, dict):
        return value.get("EnglishName") or value.get("englishName") or value.get("ArabicName") or value.get("Value") or value.get("Name")
    return value


## Authentication And DLD Calls

The request helper refreshes the iPaaS bearer token once on HTTP `401`.


In [ ]:
# @title Authentication and contract-history APIs

def get_bearer_token():
    global CURRENT_BEARER_TOKEN
    headers = {
        "Authorization": f"Basic {BASIC_AUTH}",
        "Content-Type": "application/x-www-form-urlencoded",
    }
    response = requests.post(IDS_BASE_URL, headers=headers, data={}, timeout=REQUEST_TIMEOUT_SECONDS)
    payload = response_payload(response)
    if response.status_code >= 400:
        raise RuntimeError(f"iPaaS token failed HTTP {response.status_code}: {compact_payload(payload)}")
    token = payload.get("id_token") if isinstance(payload, dict) else None
    if not token:
        raise RuntimeError(f"iPaaS token response has no id_token: {compact_payload(payload)}")
    CURRENT_BEARER_TOKEN = token
    return token


def get_active_bearer(force_refresh=False):
    global CURRENT_BEARER_TOKEN
    if force_refresh or not CURRENT_BEARER_TOKEN:
        return get_bearer_token()
    return CURRENT_BEARER_TOKEN


def request_with_bearer(method, url, headers=None, stream=False, **kwargs):
    request_headers = dict(headers or {})
    request_headers["Authorization"] = f"Bearer {get_active_bearer()}"
    kwargs.setdefault("timeout", REQUEST_TIMEOUT_SECONDS)
    response = requests.request(method, url, headers=request_headers, stream=stream, **kwargs)
    if response.status_code == 401:
        try:
            response.close()
        except Exception:
            pass
        request_headers["Authorization"] = f"Bearer {get_active_bearer(force_refresh=True)}"
        response = requests.request(method, url, headers=request_headers, stream=stream, **kwargs)
    return response


def final_auth_headers(response, fallback_headers):
    final_headers = dict(fallback_headers or {})
    sent_headers = getattr(getattr(response, "request", None), "headers", {}) or {}
    for sent_key, sent_value in sent_headers.items():
        if str(sent_key).lower() in {"authorization", "token"}:
            existing_key = next((key for key in final_headers if str(key).lower() == str(sent_key).lower()), sent_key)
            final_headers[existing_key] = sent_value
    return final_headers


def get_dld_token(emirates_id):
    bearer_token = get_active_bearer()
    auth_obj = {
        "Method": "EmiratesId",
        "MethodIdentity": str(emirates_id),
        "MethodPasscode": "",
        "DeviceKey": "MyPC",
        "ApplicationKey": "DubaiNow",
        "Platform": "Mobile",
    }
    encoded = base64.b64encode(json.dumps(auth_obj).encode()).decode()
    url = f"{DLD_BASE_URL}/generaltokenservice/1.0.0/auth"
    headers = {
        "consumer-id": CONSUMER_ID,
        "x-nv-header": encoded,
        "Content-Type": "application/json",
    }
    response = request_with_bearer("post", url, headers=headers)
    payload = response_payload(response)
    if response.status_code >= 400 or api_errors(payload):
        raise RuntimeError(f"DLD token failed for {emirates_id} HTTP {response.status_code}: {compact_payload(payload)}")
    token = payload.get("token") if isinstance(payload, dict) else None
    if not token:
        raise RuntimeError(f"DLD token response has no token for {emirates_id}: {compact_payload(payload)}")
    return token


def dld_headers(dld_token):
    return {
        "accept": "text/plain",
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
    }


def get_contract_history(dld_token):
    headers = dld_headers(dld_token)
    attempts = []
    urls = [
        ("actual", f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/properties/getcontracts"),
        ("proxy", f"{DLD_PROXY_URL}/ejariservices/properties/getcontracts"),
    ]
    for source, url in urls:
        response = request_with_bearer("get", url, headers=headers)
        payload = response_payload(response)
        result = {
            "source": source,
            "url": url,
            "status_code": response.status_code,
            "response": payload,
            "headers": final_auth_headers(response, headers),
        }
        attempts.append(result)
        if response.status_code < 400 and not api_errors(payload):
            response_data = payload.get("Response") if isinstance(payload, dict) else {}
            owner_contracts = response_data.get("OwnerContracts") if isinstance(response_data, dict) else []
            tenant_contracts = response_data.get("TenantContracts") if isinstance(response_data, dict) else []
            contracts = []
            for group, items, role in (("OwnerContracts", owner_contracts, "owner"), ("TenantContracts", tenant_contracts, "tenant")):
                for contract in items or []:
                    if isinstance(contract, dict):
                        item = dict(contract)
                        item["_history_group"] = group
                        item["_history_role"] = role
                        item["_history_source"] = source
                        contracts.append(item)
            return contracts, result, attempts
    raise RuntimeError("Contract history failed: " + " || ".join(f"{attempt['source']} HTTP {attempt['status_code']}: {compact_payload(attempt['response'], 500)}" for attempt in attempts))


## Contract Filtering And Download Checks

Status filter:

- `Active`
- `Pending`
- any status containing both `termin` and `request`

The audit streams only a response sample for success checks so it does not load every full PDF/base64 response into memory.


In [ ]:
# @title Contract filtering and download endpoint checks

def status_text(contract):
    return str(display_name(contract.get("ContractStatus") or contract.get("Status")) or contract.get("ContractStatusName") or contract.get("StatusName") or "").strip()


def normalize_status(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text or "").strip().lower()).strip()


def status_scope(contract):
    name = normalize_status(status_text(contract))
    status = contract.get("ContractStatus") if isinstance(contract, dict) else None
    identity = str(status.get("Identity") or status.get("Value") or "").strip() if isinstance(status, dict) else ""
    if name == "active":
        return "active"
    if name == "pending" or identity == "1":
        return "pending"
    if "termin" in name and "request" in name:
        return "termination_request"
    return None


def contract_row_value(contract):
    if not isinstance(contract, dict):
        return None
    return contract.get("AssociatedContractRowValue") or contract.get("ContractRowValue") or contract.get("DataRowValue") or contract.get("contract_row_value")


def contract_number(contract):
    if not isinstance(contract, dict):
        return None
    return contract.get("ContractNumber") or contract.get("contract_number")


def property_id(contract):
    if not isinstance(contract, dict):
        return None
    return contract.get("PropertyId") or contract.get("PropertyID")


def contract_title(contract):
    if not isinstance(contract, dict):
        return None
    return display_name(contract.get("Title") or contract.get("PropertyNumber") or contract.get("PropertyName")) or contract_number(contract) or contract_row_value(contract)


def consolidate_in_scope(contracts):
    by_row = {}
    missing_row_count = 0
    for contract in contracts:
        scope = status_scope(contract)
        if not scope:
            continue
        row_value = contract_row_value(contract)
        if not row_value:
            missing_row_count += 1
            continue
        if row_value not in by_row:
            item = dict(contract)
            item["_status_scope"] = scope
            item["_source_groups"] = [contract.get("_history_group")]
            item["_roles"] = [contract.get("_history_role")]
            by_row[row_value] = item
        else:
            existing = by_row[row_value]
            if contract.get("_history_group") not in existing["_source_groups"]:
                existing["_source_groups"].append(contract.get("_history_group"))
            if contract.get("_history_role") not in existing["_roles"]:
                existing["_roles"].append(contract.get("_history_role"))
            if existing.get("_status_scope") == "pending" and scope in {"active", "termination_request"}:
                existing["_status_scope"] = scope
    return list(by_row.values()), missing_row_count


def parse_sample_payload(sample_bytes):
    if not sample_bytes:
        return None
    try:
        text = sample_bytes.decode("utf-8", errors="replace")
    except Exception:
        return None
    stripped = text.strip()
    if not stripped:
        return ""
    if stripped.startswith("{") or stripped.startswith("["):
        try:
            return decode_json_like(json.loads(stripped))
        except Exception:
            return stripped
    return stripped


def sample_response(response, max_bytes=STREAM_SAMPLE_BYTES):
    chunks = []
    total = 0
    try:
        for chunk in response.iter_content(chunk_size=65536):
            if not chunk:
                continue
            chunks.append(chunk)
            total += len(chunk)
            if total >= max_bytes:
                break
    finally:
        response.close()
    return b"".join(chunks), total


def sample_has_contract_file(payload_or_text):
    if isinstance(payload_or_text, dict):
        keys = ("ContractFile", "contractFile", "FileContent", "fileContent", "Document", "document", "Pdf", "pdf", "ReportData", "reportData")
        for key in keys:
            if payload_or_text.get(key):
                return True
        response_data = payload_or_text.get("Response")
        if isinstance(response_data, dict):
            return any(bool(response_data.get(key)) for key in keys)
        return isinstance(response_data, str) and bool(response_data.strip())
    if isinstance(payload_or_text, str):
        text = payload_or_text.strip()
        return bool(text) and ("ReportData" in text or "ContractFile" in text or len(text) > 200)
    return False


def download_url(api_name, row_value):
    endpoint = "downloadTenancyContract" if api_name == "downloadTenancyContract" else "download"
    return f"{DLD_PROXY_URL}/ejariservices/contracts/{row_value}/{endpoint}"


def call_download(api_name, row_value, dld_token):
    url = download_url(api_name, row_value)
    headers = dld_headers(dld_token)
    try:
        response = request_with_bearer("get", url, headers=headers, stream=True)
        sent_headers = final_auth_headers(response, headers)
        content_type = response.headers.get("Content-Type", "")
        content_length = response.headers.get("Content-Length")
        sample, sample_bytes = sample_response(response)
        payload = parse_sample_payload(sample)
        errors = api_errors(payload)

        non_json_success_body = response.status_code < 400 and sample_bytes > 0 and "json" not in content_type.lower()
        json_or_text_success_body = response.status_code < 400 and sample_has_contract_file(payload)
        success = response.status_code < 400 and not errors and (non_json_success_body or json_or_text_success_body)

        if success:
            failure_reason = ""
        elif response.status_code >= 400:
            failure_reason = f"HTTP {response.status_code}"
        elif errors:
            failure_reason = " | ".join(errors)
        else:
            failure_reason = "No downloaded contract file/body detected in response sample."

        response_for_log = payload
        if isinstance(response_for_log, str) and len(response_for_log) > 5000:
            response_for_log = response_for_log[:5000] + "... [truncated sample]"

        return {
            "api_name": api_name,
            "method": "get",
            "url": url,
            "headers": sent_headers,
            "status_code": response.status_code,
            "content_type": content_type,
            "content_length": content_length,
            "sample_bytes": sample_bytes,
            "response": response_for_log,
            "success": success,
            "failure_reason": failure_reason,
        }
    except Exception as exc:
        return {
            "api_name": api_name,
            "method": "get",
            "url": url,
            "headers": headers,
            "status_code": None,
            "content_type": "",
            "content_length": "",
            "sample_bytes": 0,
            "response": None,
            "success": False,
            "failure_reason": str(exc),
        }


## Failure Artifacts And Report Writers

Every failed endpoint check gets:

- `failed_<api>_... .sh`
- `failed_<api>_... _response.json`

The summary CSV/JSON files are rewritten after each Emirates ID so partial results survive interruptions.


In [ ]:
# @title Failure artifacts and report writers
summary_rows = []
download_check_rows = []
download_failure_rows = []

SUMMARY_COLUMNS = [
    "emirates_id",
    "history_source",
    "history_total_rows",
    "in_scope_unique_contracts",
    "active_contracts",
    "pending_contracts",
    "termination_request_contracts",
    "download_checks",
    "download_failures",
    "contracts_missing_row_value",
    "elapsed_seconds",
    "error",
]
CHECK_COLUMNS = [
    "emirates_id",
    "api_name",
    "success",
    "failure_reason",
    "http_status_code",
    "content_type",
    "content_length",
    "sample_bytes",
    "contract_row_value",
    "contract_number",
    "contract_status",
    "status_scope",
    "history_groups",
    "roles",
    "property_id",
    "title",
    "url",
    "curl_path",
    "response_path",
]


def save_failure_artifacts(emirates_id, contract, result):
    row_value = contract_row_value(contract)
    stem = (
        f"failed_{safe_filename(result['api_name'])}_{safe_filename(emirates_id)}_"
        f"{safe_filename(property_id(contract))}_{safe_filename(row_value)}_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
    )
    curl_path = FAILURE_DIR / f"{stem}.sh"
    response_path = FAILURE_DIR / f"{stem}_response.json"
    curl_path.write_text(build_curl(result["method"], result["url"], result.get("headers")), encoding="utf-8")

    detail = {
        "timestamp": datetime.now().isoformat(),
        "emirates_id": emirates_id,
        "api_name": result["api_name"],
        "method": result["method"],
        "url": result["url"],
        "status_code": result["status_code"],
        "content_type": result.get("content_type"),
        "content_length": result.get("content_length"),
        "sample_bytes": result.get("sample_bytes"),
        "failure_reason": result.get("failure_reason"),
        "contract_row_value": row_value,
        "contract_number": contract_number(contract),
        "contract_status": status_text(contract),
        "status_scope": contract.get("_status_scope"),
        "history_groups": contract.get("_source_groups") or [contract.get("_history_group")],
        "roles": contract.get("_roles") or [contract.get("_history_role")],
        "property_id": property_id(contract),
        "title": contract_title(contract),
        "response": result.get("response"),
    }
    response_path.write_text(json.dumps(detail, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return str(curl_path), str(response_path)


def write_csv(path, rows, columns):
    with Path(path).open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def write_outputs():
    write_csv(RUN_DIR / "summary.csv", summary_rows, SUMMARY_COLUMNS)
    write_csv(RUN_DIR / "download_checks.csv", download_check_rows, CHECK_COLUMNS)
    write_csv(RUN_DIR / "download_failures.csv", download_failure_rows, CHECK_COLUMNS)
    (RUN_DIR / "summary.json").write_text(json.dumps(summary_rows, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    (RUN_DIR / "download_checks.json").write_text(json.dumps(download_check_rows, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    (RUN_DIR / "download_failures.json").write_text(json.dumps(download_failure_rows, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


## Run The Audit

This cell executes the audit and prints an Emirates-ID-wise summary.


In [ ]:
# @title Run contract download audit
print("Run folder:", RUN_DIR)

for emirates_id in OWNER_EMIRATES_IDS:
    if master_selection != "all":
        decision = ask_choice(
            f"Process Emirates ID {emirates_id}?",
            [("yes", "Yes"), ("no", "No")],
            default="yes",
            title="Process Emirates ID",
        )
        if decision != "yes":
            print(f"\nSkipping Emirates ID {emirates_id}")
            continue

    print(f"\nProcessing Emirates ID {emirates_id}")
    started_at = time.time()
    try:
        CURRENT_BEARER_TOKEN = None
        get_active_bearer(force_refresh=True)
        dld_token = get_dld_token(emirates_id)
        contracts, history_result, _ = get_contract_history(dld_token)
        in_scope_contracts, missing_row_count = consolidate_in_scope(contracts)

        counts = {"active": 0, "pending": 0, "termination_request": 0}
        for contract in in_scope_contracts:
            counts[contract.get("_status_scope")] = counts.get(contract.get("_status_scope"), 0) + 1

        download_checks = 0
        download_failures = 0

        for contract in in_scope_contracts:
            row_value = contract_row_value(contract)
            api_names = ["downloadTenancyContract"]
            if contract.get("_status_scope") in {"active", "termination_request"}:
                api_names.append("download")

            for api_name in api_names:
                result = call_download(api_name, row_value, dld_token)
                download_checks += 1
                row = {
                    "emirates_id": emirates_id,
                    "api_name": api_name,
                    "success": result.get("success"),
                    "failure_reason": result.get("failure_reason"),
                    "http_status_code": result.get("status_code"),
                    "content_type": result.get("content_type"),
                    "content_length": result.get("content_length"),
                    "sample_bytes": result.get("sample_bytes"),
                    "contract_row_value": row_value,
                    "contract_number": contract_number(contract),
                    "contract_status": status_text(contract),
                    "status_scope": contract.get("_status_scope"),
                    "history_groups": ";".join(contract.get("_source_groups") or [contract.get("_history_group") or ""]),
                    "roles": ";".join(contract.get("_roles") or [contract.get("_history_role") or ""]),
                    "property_id": property_id(contract),
                    "title": contract_title(contract),
                    "url": result.get("url"),
                    "curl_path": "",
                    "response_path": "",
                }

                if not result.get("success"):
                    download_failures += 1
                    curl_path, response_path = save_failure_artifacts(emirates_id, contract, result)
                    row["curl_path"] = curl_path
                    row["response_path"] = response_path
                    download_failure_rows.append(row.copy())
                    print(f"  FAIL {api_name} | {row_value} | {result.get('failure_reason')} | {curl_path}")

                download_check_rows.append(row)

        summary_rows.append({
            "emirates_id": emirates_id,
            "history_source": history_result["source"],
            "history_total_rows": len(contracts),
            "in_scope_unique_contracts": len(in_scope_contracts),
            "active_contracts": counts.get("active", 0),
            "pending_contracts": counts.get("pending", 0),
            "termination_request_contracts": counts.get("termination_request", 0),
            "download_checks": download_checks,
            "download_failures": download_failures,
            "contracts_missing_row_value": missing_row_count,
            "elapsed_seconds": round(time.time() - started_at, 2),
            "error": "",
        })
        print(
            f"  done: in_scope={len(in_scope_contracts)} checks={download_checks} failures={download_failures} "
            f"active={counts.get('active', 0)} pending={counts.get('pending', 0)} termination_request={counts.get('termination_request', 0)}"
        )
    except Exception as exc:
        summary_rows.append({
            "emirates_id": emirates_id,
            "history_source": "",
            "history_total_rows": 0,
            "in_scope_unique_contracts": 0,
            "active_contracts": 0,
            "pending_contracts": 0,
            "termination_request_contracts": 0,
            "download_checks": 0,
            "download_failures": 1,
            "contracts_missing_row_value": 0,
            "elapsed_seconds": round(time.time() - started_at, 2),
            "error": str(exc),
        })
        print(f"  ERROR: {exc}")
    finally:
        write_outputs()

print("\nDONE")
print("Run folder:", RUN_DIR)
print("Total download checks:", len(download_check_rows))
print("Total failures:", len(download_failure_rows))
for row in summary_rows:
    print(
        f"{row['emirates_id']}: in_scope={row['in_scope_unique_contracts']} checks={row['download_checks']} "
        f"failures={row['download_failures']} active={row['active_contracts']} pending={row['pending_contracts']} "
        f"termination_request={row['termination_request_contracts']} error={row['error']}"
    )


## Output Files And Interpretation

The run folder contains three main report files:

- `summary.csv` has one row per Emirates ID with total contract-history rows, in-scope unique contracts, download checks, and failures.
- `download_checks.csv` has one row per endpoint call. Use this when you need to prove which contracts and endpoints were tested.
- `download_failures.csv` contains only failed endpoint checks and includes `curl_path` plus `response_path` for API-team follow-up.

A failure curl and response file should be shared as a pair. Curl files can contain live request headers, so treat them as sensitive evidence.


## Preview Failure Rows

If there are failures, this cell displays the failure table. If there are no failures, it prints a clean result message.


In [ ]:
# @title Preview failures
try:
    import pandas as pd
    if download_failure_rows:
        display(pd.DataFrame(download_failure_rows, columns=CHECK_COLUMNS))
    else:
        print("No download endpoint failures found.")
    display(pd.DataFrame(summary_rows, columns=SUMMARY_COLUMNS))
except Exception:
    if download_failure_rows:
        print(json.dumps(download_failure_rows, ensure_ascii=False, indent=2, default=str))
    else:
        print("No download endpoint failures found.")
    print(json.dumps(summary_rows, ensure_ascii=False, indent=2, default=str))
